In [1]:
%%writefile extractor.py
import re
from datetime import date, timedelta
import pandas as pd

MONTHS = {"january":1,"jan":1,"february":2,"feb":2,"march":3,"april":4,"may":5,
          "june":6,"july":7,"august":8,"september":9,"october":10,"oct":10,
          "november":11,"nov":11,"december":12,"dec":12}
KEYWORDS = ["meeting","quiz","exam","bill","assignment","class","submit",
            "deadline","presentation","lab"]
RELATIVE = {"ajke":0,"aj":0,"kal":1,"kalke":1,"kalle":1,"porshu":2}
COLUMNS = ["task", "due", "needs_confirmation", "source"]

def extract(text, today):
    rows = []
    for clause in re.split(r",| ar | and ", text.lower()):
        clause = clause.strip()
        task = next((k for k in KEYWORDS if k in clause), None)
        if not clause or not task:
            continue
        due, confirm = None, True
        m = re.search(r"(\d{1,2})\s*(?:th|st|nd|rd)?\s+([a-z]+)", clause)
        words = clause.split()
        if m and m.group(2) in MONTHS:
            try:
                d = date(today.year, MONTHS[m.group(2)], int(m.group(1)))
                if d < today:
                    d = d.replace(year=today.year + 1)
                due, confirm = d, False
            except ValueError:
                pass
        elif any(w in RELATIVE for w in words):
            w = next(w for w in words if w in RELATIVE)
            due, confirm = today + timedelta(days=RELATIVE[w]), False
        elif "porer shoptaho" in clause or "next week" in clause:
            due, confirm = today + timedelta(days=7), True
        rows.append({"task": task, "due": due,
                     "needs_confirmation": confirm, "source": clause})
    return pd.DataFrame(rows, columns=COLUMNS)

Writing extractor.py


In [5]:
from datetime import date
from extractor import extract

TODAY = date(2026, 10, 18)
extract("Amr kalle meeting ache, 20 October quiz, ar electricity bill porer shoptaho due", TODAY)

,task,due,needs_confirmation,source
0,meeting,2026-10-19,False,amr kalle meeting ache
1,quiz,2026-10-20,False,20 october quiz
2,bill,2026-10-25,True,electricity bill porer shoptaho due


In [6]:
import pandas as pd

tests = [
    # (sentence, expected_task, expected_date)
    ("Amr kalle meeting ache",              "meeting",      date(2026, 10, 19)),
    ("20 October quiz",                     "quiz",         date(2026, 10, 20)),
    ("electricity bill porer shoptaho due", "bill",         date(2026, 10, 25)),
    ("porshu assignment submit",            "assignment",   date(2026, 10, 20)),
    ("ajke lab ache",                       "lab",          date(2026, 10, 18)),
    ("kalke quiz ache",                     "quiz",         date(2026, 10, 19)),
    ("25 October exam",                     "exam",         date(2026, 10, 25)),
    ("5 November deadline",                 "deadline",     date(2026, 11, 5)),
    ("next week presentation",              "presentation", date(2026, 10, 25)),
    ("porer mongolbar meeting",             "meeting",      date(2026, 10, 20)),
    # ADD YOUR OWN SENTENCES HERE, one per line
]

results = []
for sentence, exp_task, exp_date in tests:
    out = extract(sentence, TODAY)
    ok = ((out["task"] == exp_task) & (out["due"] == exp_date)).any() if len(out) else False
    got = f"{out.iloc[0]['task']} / {out.iloc[0]['due']}" if len(out) else "nothing found"
    results.append({"sentence": sentence, "expected": f"{exp_task} / {exp_date}",
                    "got": got, "correct": ok})

res = pd.DataFrame(results)
display(res)
print(f"Accuracy: {res['correct'].sum()} of {len(res)} = {res['correct'].mean():.0%}")

,sentence,expected,got,correct
0,Amr kalle meeting ache,meeting / 2026-10-19,meeting / 2026-10-19,True
1,20 October quiz,quiz / 2026-10-20,quiz / 2026-10-20,True
2,electricity bill porer shoptaho due,bill / 2026-10-25,bill / 2026-10-25,True
3,porshu assignment submit,assignment / 2026-10-20,assignment / 2026-10-20,True
4,ajke lab ache,lab / 2026-10-18,lab / 2026-10-18,True
5,kalke quiz ache,quiz / 2026-10-19,quiz / 2026-10-19,True
6,25 October exam,exam / 2026-10-25,exam / 2026-10-25,True
7,5 November deadline,deadline / 2026-11-05,deadline / 2026-11-05,True
8,next week presentation,presentation / 2026-10-25,presentation / 2026-10-25,True
9,porer mongolbar meeting,meeting / 2026-10-20,meeting / None,False


Accuracy: 9 of 10 = 90%


In [8]:
tests = [
 # --- ajke / kal / porshu (today, tomorrow, day after) ---
 ("ajke lab ache", "lab", date(2026,10,18)),
 ("kal class ache", "class", date(2026,10,19)),
 ("kalke quiz ache", "quiz", date(2026,10,19)),
 ("porshu assignment submit korte hobe", "assignment", date(2026,10,20)),
 ("amr kalle meeting ache", "meeting", date(2026,10,19)),
 ("ajke presentation ache", "presentation", date(2026,10,18)),
 ("porshu exam", "exam", date(2026,10,20)),
 ("ajke electricity bill dite hobe", "bill", date(2026,10,18)),
 ("kal gas bill er last date", "bill", date(2026,10,19)),
 ("porshu deadline", "deadline", date(2026,10,20)),
 # --- exact dates ---
 ("20 October quiz", "quiz", date(2026,10,20)),
 ("25 October exam", "exam", date(2026,10,25)),
 ("5 November deadline", "deadline", date(2026,11,5)),
 ("28 Oct assignment jomar last date", "assignment", date(2026,10,28)),
 ("10 November class test", "class", date(2026,11,10)),
 ("12 Nov meeting", "meeting", date(2026,11,12)),
 ("1 December exam shuru", "exam", date(2026,12,1)),
 ("30 October presentation", "presentation", date(2026,10,30)),
 ("22nd October lab", "lab", date(2026,10,22)),
 ("electricity bill 15 November er moddhe", "bill", date(2026,11,15)),
 ("December 5 exam", "exam", date(2026,12,5)),
 # --- next week ---
 ("next week presentation", "presentation", date(2026,10,25)),
 ("electricity bill porer shoptaho", "bill", date(2026,10,25)),
 ("porer shoptaho quiz ache", "quiz", date(2026,10,25)),
 ("next week exam shuru", "exam", date(2026,10,25)),
 ("internet bill porer shoptaho due", "bill", date(2026,10,25)),
 # --- weekdays ---
 ("porer mongolbar meeting", "meeting", date(2026,10,20)),
 ("shukrobar class ache", "class", date(2026,10,23)),
 ("brihoshpotibar quiz", "quiz", date(2026,10,22)),
 ("shonibar lab", "lab", date(2026,10,24)),
 ("this Thursday exam", "exam", date(2026,10,22)),
 # --- with times ---
 ("kal 3 tar shomoy meeting", "meeting", date(2026,10,19)),
 ("porshu shokal 9 tay exam", "exam", date(2026,10,20)),
 ("ajke raat 8 tay class", "class", date(2026,10,18)),
 # --- natural mixed Banglish ---
 ("amr kal quiz ache, ektu porte hobe", "quiz", date(2026,10,19)),
 ("Ami 24 October meeting e jabo", "meeting", date(2026,10,24)),
 ("exam er date 12 November", "exam", date(2026,11,12)),
 ("bill pay korte hobe 20 October er moddhe", "bill", date(2026,10,20)),
 ("assignment jomar last date 3 November", "assignment", date(2026,11,3)),
 ("kal sokale class ache", "class", date(2026,10,19)),
 # --- everyday Bangladeshi tasks (words the extractor does not know) ---
 ("porshu viva ache", "viva", date(2026,10,20)),
 ("kal tuition ache", "tuition", date(2026,10,19)),
 ("bari bhara dite hobe 25 October", "rent", date(2026,10,25)),
 ("ajke doctor appointment", "doctor", date(2026,10,18)),
 ("5 November biyer dawat", "wedding", date(2026,11,5)),
 ("wifi bill kal", "bill", date(2026,10,19)),
 # --- Bangla script ---
 ("কাল ক্লাস আছে", "class", date(2026,10,19)),
 ("পরশু পরীক্ষা", "exam", date(2026,10,20)),
 ("আজ মিটিং আছে", "meeting", date(2026,10,18)),
 ("২০ অক্টোবর কুইজ", "quiz", date(2026,10,20)),
]